In [1]:
import yfinance as yf
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.pyplot import Figure as fig
from pylab import rcParams
import datetime
from datetime import timedelta
import statsmodels.api as sm
rcParams['figure.figsize'] = 20,10
from ResearchClass import PlotEvaluations, EvaluationMetrics, TradesBook, StrategyTemplate
import os

pd.options.mode.chained_assignment = None  # default='warn'

In [2]:
class RSIIndicator(StrategyTemplate):
    """
    RSI (Relative Strength Index) strategy class that inherits from StrategyTemplate.
    Calculates RSI and implements signals based on it.
    """
    def __init__(self, data, start_date, end_date, interval, indicator_parameters):
        self.data=data
        self.indicator_parameters=indicator_parameters

    def AddIndicators(self):
        """
        Adds RSI indicators to the data.
        RSI is calculated as:
        RSI = 100 - (100 / (1 + RS)),
        where RS is the average gain / average loss over a lookback period.
        """
       
        LOOKBACK_PERIOD= self.indicator_parameters[0]
        # Calculate price differences
        self.data['Price_Change'] = self.data['Close'].diff()

        # Separate gains and losses
        self.data['Gain'] = np.where(self.data['Price_Change'] > 0, self.data['Price_Change'], 0)
        self.data['Loss'] = np.where(self.data['Price_Change'] < 0, -self.data['Price_Change'], 0)

        # Calculate average gain and loss
        self.data['Avg_Gain'] = self.data['Gain'].rolling(LOOKBACK_PERIOD).mean()
        self.data['Avg_Loss'] = self.data['Loss'].rolling(LOOKBACK_PERIOD).mean()

        # Calculate RS and RSI
        self.data['RS'] = self.data['Avg_Gain'] / self.data['Avg_Loss']
        self.data['RSI'] = 100 - (100 / (1 + self.data['RS']))

    def strategyLogic(self, TradeBook, row, idx):
        """
        RSI-based trading logic.
        - Buy when RSI < 30 (oversold).
        - Sell when RSI > 70 (overbought).
        """
        if idx < self.indicator_parameters[0]:
            return

        if TradeBook.CurrentSizing() == 0:
            if row['RSI'] < 30:  # Oversold signal
                TradeBook.OpenTrade(row['Open'], 1, row['Open'] * (1 - self.indicator_parameters[1]), idx)
            elif row['RSI'] > 70:  # Overbought signal
                TradeBook.OpenTrade(row['Open'], -1, row['Open'] * (1 + self.indicator_parameters[1]), idx)
